# Lab 9: Combining Context Providers with Memory Tools

In this lab, you will add memory tools to the passive context provider, giving the agent explicit control over searching, saving, and recalling memories on demand. The context provider handles background injection automatically, while tools let the agent actively decide to save a preference or search for a specific entity — together they form a complete memory system.

## What you will learn

- How to create memory tools with `create_memory_tools()`
- The combined pattern: passive context provider + active tools
- How the agent uses tools like `remember_preference` and `recall_preferences`
- How to verify stored memories after a conversation

## Setup

In [1]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

provider_name = os.getenv("LLM_PROVIDER", "openai").lower()
if provider_name == "azure":
    assert os.getenv("AZURE_OPENAI_API_KEY"), "AZURE_OPENAI_API_KEY not set in .env file"
    assert os.getenv("AZURE_OPENAI_ENDPOINT"), "AZURE_OPENAI_ENDPOINT not set in .env file"
else:
    assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not set in .env file"
assert os.getenv("NEO4J_URI"), "NEO4J_URI not set in .env file"
print("Environment loaded successfully")

Environment loaded successfully


## Import Dependencies

In [2]:
from pydantic import SecretStr

from llm_provider import get_client, get_provider

from neo4j_agent_memory import MemoryClient, MemorySettings
from neo4j_agent_memory.integrations.microsoft_agent import (
    Neo4jMicrosoftMemory,
    create_memory_tools,
)

## Configure and Connect

In [3]:
if get_provider() == "azure":
    llm_model = os.getenv("AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME", "gpt-5-mini")
else:
    llm_model = os.getenv("OPENAI_RESPONSES_MODEL_ID", "gpt-5-mini")

settings = MemorySettings(
    neo4j={
        "uri": os.environ["NEO4J_URI"],
        "username": os.environ["NEO4J_USERNAME"],
        "password": SecretStr(os.environ["NEO4J_PASSWORD"]),
    },
    extraction={
        "enable_gliner": False,  # Disabled: downloads ~500MB model from HuggingFace, impractical in a workshop
        "llm_model": llm_model,
        "llm_temperature": 1.0,  # gpt-5-mini only supports temperature=1.0
    },
)

memory_client = MemoryClient(settings)
await memory_client.connect()
print("Connected to Neo4j")

Connected to Neo4j


## Create Memory and Tools

Context providers handle passive injection (runs automatically every turn), while tools handle active operations (the agent chooses to call them). Together they give the agent both background awareness and targeted memory control. Build on the previous lab by adding memory tools:

1. Create `Neo4jMicrosoftMemory` as before
2. Call `create_memory_tools(memory)` to generate callable tool functions
3. Pass both `tools=tools` and `context_providers=[memory.context_provider]` to the agent

The agent instructions should tell it to use `remember_preference` when a user expresses a preference, and `recall_preferences` when making recommendations.

In [4]:
memory = Neo4jMicrosoftMemory.from_memory_client(
    memory_client=memory_client,
    session_id="movie-tools-session",
    include_short_term=True,
    include_long_term=True,
    extract_entities=True,
)

tools = create_memory_tools(memory)

for tool in tools:
    print(f"  - {tool.name}: {tool.description[:60]}...")

client = get_client()

agent = client.as_agent(
    name="movie-assistant",
    instructions=(
        "You are a movie recommendation assistant with persistent memory.\n\n"
        "IMPORTANT: You MUST call the remember_preference tool every time "
        "a user states a preference, favorite, or interest. Do NOT just "
        "acknowledge it verbally — you must actually invoke the tool to "
        "persist it. For example, if a user says they love sci-fi, call "
        "remember_preference with category='genre' and preference='loves sci-fi'.\n\n"
        "When making recommendations, use recall_preferences to check "
        "what the user likes before suggesting something.\n\n"
        "You have access to a knowledge graph of movies and "
        "your memory of past conversations."
    ),
    tools=tools,
    context_providers=[memory.context_provider],
)

  - search_memory: Search the user's memory for relevant information including ...
  - remember_preference: Save a user preference for future reference. Use this when t...
  - recall_preferences: Recall user preferences related to a topic or category. Use ...
  - search_knowledge: Search the knowledge graph for entities (products, brands, c...
  - remember_fact: Save a factual statement for future reference. Use this for ...
  - find_similar_tasks: Find similar tasks from past interactions to learn from prev...


## Run a Conversation

Test the combined pattern with a 3-turn conversation:

1. **Turn 1** — User expresses preferences. The agent should call `remember_preference`.
2. **Turn 2** — User asks about their preferences. The agent should call `recall_preferences`.
3. **Turn 3** — Both passive context and active tools contribute to a recommendation.

In [5]:
session = agent.create_session()

queries = [
    "I love Christopher Nolan movies and anything about space.",
    "What are my movie preferences?",
    "Based on what you know about me, what should I watch next?",
]

for query in queries:
    print(f"\nUser: {query}")
    async for update in agent.run(query, stream=True, session=session):
        if update.text:
            print(update.text, end="", flush=True)
    print()


User: I love Christopher Nolan movies and anything about space.
Great — thanks, I’ve saved those preferences.

Top picks based on liking Christopher Nolan + space movies (and remembering you enjoy sci‑fi/time‑travel):

1. Interstellar (2014) — Nolan’s space epic; strong science, emotional core, relativity/time‑dilation themes that tie into your time‑travel interest. First choice.
2. 2001: A Space Odyssey (1968) — Classic, philosophical, visually stunning. If you like big-picture cosmic stories.
3. The Martian (2015) — Smart, grounded survival story with a lighter, hopeful tone and strong science.
4. Gravity (2013) — Tense, immersive one‑ship survival/space experience with incredible visuals and sound design.
5. Moon (2009) — Intimate, eerie, character‑driven near‑future lunar story with psychological twists.
6. Sunshine (2007) — Blend of sci‑fi spectacle and psychological suspense on a mission to save the Sun.
7. Ad Astra (2019) — Introspective, slow‑burn space drama about a man’s sea

## Verify Stored Memories

Inspect preferences, entities, and messages stored by the combined approach.

In [6]:
results = await memory.search_memory(
    query="Christopher Nolan movies space",
    include_messages=True,
    include_entities=True,
    include_preferences=True,
    limit=5,
)

print("=== Stored Memories ===\n")

if results.get("preferences"):
    print("Preferences:")
    for pref in results["preferences"]:
        print(f"  [{pref['category']}] {pref['preference']}")
    print()

if results.get("entities"):
    print("Entities:")
    for entity in results["entities"][:5]:
        print(f"  {entity['name']} ({entity['type']})")
    print()

if results.get("messages"):
    print(f"Messages stored: {len(results['messages'])}")

=== Stored Memories ===

Preferences:
  [director] loves Christopher Nolan
  [topic] loves space movies

Entities:
  Inception (OBJECT)

Messages stored: 2


## Close the Connection

In [7]:
await memory_client.close()
print("Connection closed")

Connection closed


## Summary

Memory tools give the agent explicit control over search, save, and recall operations, while the context provider continues to handle background memory injection automatically. The combined pattern covers both passive context (always available) and active operations (agent-directed).

**What's next:** In the next lab, you will explore reasoning memory to capture and learn from past agent executions.